In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

In [2]:
model_id = "meta-llama/Llama-Guard-3-8B"
device = "cuda"
dtype = torch.bfloat16

In [ ]:
import re
import pandas as pd

class TextPreprocessor:
    CLEAN_RE = re.compile(r"[^a-z0-9]+")

    def __init__(self, dataset_path: str):
        self.dataset_path = dataset_path
        self._dataset = None
        self._processed_df = None

    def load_dataset(self):
        if self._dataset is None:
            try:
                self._dataset = pd.read_excel(self.dataset_path)
            except Exception as e:
                raise RuntimeError(f"Failed to load dataset: {e}")
        return self._dataset

    def handle_dataset(self):
        if self._processed_df is not None:
            return self._processed_df

        df = self.load_dataset()

        query_df = df[['query', 'Toxic Category']].dropna().copy()
        query_df.columns = ['text_content', 'labels']

        desc_df = df[['image descriptions', 'Toxic Category']].dropna().copy()
        desc_df = (
              desc_df.groupby('image descriptions')['Toxic Category']
              .agg(lambda x: x.mode()[0])
              .reset_index()
          )
        desc_df.columns = ['text_content', 'labels']

        combined_df = pd.concat([query_df, desc_df], ignore_index=True)
        self._processed_df = combined_df
        return combined_df

    def clean_text(self, text: str) -> str:
        if not isinstance(text, str):
            return ""

        text = text.lower()
        text = self.CLEAN_RE.sub(" ", text)
        return text.strip()

    def clean_batch(self):
        df = self.handle_dataset()
        df.drop_duplicates(inplace=True)

        df = df.copy()
        df['text_content'] = df['text_content'].apply(self.clean_text)
        return df

    def get_data_info(self):
        df = self.handle_dataset()

        info = {
            "dataset_size": len(df),
            "label_distribution": df['labels'].value_counts().to_dict(),
            "missing_values": df.isnull().sum().to_dict()
        }

        return info

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

: 

In [ ]:
dataset_path = "/content/drive/My Drive/NLP_Neurova_toxic_content_classification.xlsx"
text_preprocessor = TextPreprocessor(dataset_path)
df = text_preprocessor.clean_batch()
df.iloc[-1]

In [ ]:
CATEGORIES =['Child Sexual Exploitation' 'Elections' 'Non-Violent Crimes' 'Safe'
 'Sex-Related Crimes' 'Suicide & Self-Harm' 'Unknown S-Type'
 'Violent Crimes' 'unsafe']

In [4]:
taxonomy = """Task: Check if there is unsafe content in 'User' messages according to our safety policy.

<BEGIN UNSAFE CONTENT CATEGORIES>
S1: Child Sexual Exploitation. 
S2: Elections.
S3: Non-Violent Crimes.
S4: Safe.
S5: Sex-Related Crimes.
S6: Suicide & Self-Harm.
S7: Unknown S-Type.
S8: Violent Crimes.
S9: unsafe.
<END UNSAFE CONTENT CATEGORIES>

Output Format:
Line 1: safe or unsafe
Line 2: If unsafe, the comma-separated list of category codes (e.g., S1, S2)
"""

In [ ]:
from groq import Groq

client = Groq(api_key="your_api_key")

def moderate_content(user_text):
    completion = client.chat.completions.create(
        model="llama-guard-3-8b",
        messages=[
            {"role": "system", "content": taxonomy},
            {"role": "user", "content": user_text}
        ]
    )
    return completion.choices[0].message.content

# Test it
print(moderate_content("I will find where you live and hurt you!"))
# Output should be:
# unsafe
# S1